In [0]:
# SETUP

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import TimestampType, DateType, DoubleType
from datetime import datetime, date, timedelta

# ── Catalog / Schema ──────────────────────────────────────────────────────────
CATALOG       = "bfsi"
BRONZE_SCHEMA = "bronze_layer"
SILVER_SCHEMA = "silver_layer"
GOLD_SCHEMA   = "gold_layer"

# ── Silver source tables ──────────────────────────────────────────────────────
UNIFIED_TXN_TABLE    = f"{CATALOG}.{SILVER_SCHEMA}.unified_transactions"
TXN_FEATURES_TABLE   = f"{CATALOG}.{SILVER_SCHEMA}.transaction_features"
FRAUD_ALERTS_TABLE   = f"{CATALOG}.{SILVER_SCHEMA}.fraud_alerts"
DIM_CUSTOMER_TABLE   = f"{CATALOG}.{SILVER_SCHEMA}.dim_customer_risk"
CUSTOMERS_MASKED     = f"{CATALOG}.{SILVER_SCHEMA}.customers_masked"

# ── Gold target tables ────────────────────────────────────────────────────────
SAR_TABLE            = f"{CATALOG}.{GOLD_SCHEMA}.sar_report_feed"
RISK_METRICS_TABLE   = f"{CATALOG}.{GOLD_SCHEMA}.daily_risk_metrics"
CAPITAL_TABLE        = f"{CATALOG}.{GOLD_SCHEMA}.capital_adequacy_summary"

# ── Basel III constants ───────────────────────────────────────────────────────
LGD_UNSECURED        = 0.45   # Loss Given Default — unsecured loans
LGD_SECURED          = 0.25   # Loss Given Default — secured loans
PD_LOW               = 0.005  # Probability of Default — LOW risk
PD_MEDIUM            = 0.03   # Probability of Default — MEDIUM risk
PD_HIGH              = 0.12   # Probability of Default — HIGH risk
RWA_MULTIPLIER       = 12.5   # Basel III standard multiplier
RBI_MIN_CAR          = 11.5   # RBI minimum Capital Adequacy Ratio (%)

# ── Reporting config ──────────────────────────────────────────────────────────
REPORT_DATE          = date.today()
SAR_THRESHOLD_INR    = 1_000_000.0   # ₹10,00,000 single-day threshold

# ── Job metadata ─────────────────────────────────────────────────────────────
JOB_TS     = datetime.now()
JOB_TS_STR = JOB_TS.isoformat()

print("=" * 65)
print(f"  Gold Layer Pipeline — NovaCred Bank")
print(f"  Report Date : {REPORT_DATE}")
print(f"  Timestamp   : {JOB_TS_STR}")
print("=" * 65)

---
## Task 3.1 — SAR Report Feed (`gold_layer.sar_report_feed`)

**Regulatory requirement:** RBI mandates that all Suspicious Activity Reports  
be ready by **06:50 AM daily** for compliance system ingestion.

A transaction qualifies for SAR inclusion if **either** condition is met:
1. The customer's **total transaction value on a single day ≥ ₹10,00,000**
2. The transaction was **flagged by any fraud rule** in Task 2.4

**Zero PII policy:** Only masked/tokenized identifiers are written.  
Source columns used: `unified_transactions` + `fraud_alerts` + `dim_customer_risk`

In [0]:
# ── Step 1: Identify customers breaching ₹10L single-day threshold ────────────
daily_customer_totals = (
    spark.table(UNIFIED_TXN_TABLE)
    .filter(F.col("status") == "SUCCESS")
    .withColumn("txn_date", F.to_date("txn_ts"))
    .groupBy("customer_id", "txn_date")
    .agg(F.sum("txn_amount_inr").alias("daily_total_inr"))
    .filter(F.col("daily_total_inr") >= SAR_THRESHOLD_INR)
    .select("customer_id", "txn_date")
)

high_value_customers_count = daily_customer_totals.count()
print(f"📊 Customer-days breaching ₹10L threshold: {high_value_customers_count:,}")
# daily_customer_totals.cache()

In [0]:
# ── Step 2: Pull all transactions for high-value customers on breach day ───────
unified_df  = spark.table(UNIFIED_TXN_TABLE)
fraud_df    = spark.table(FRAUD_ALERTS_TABLE)

# Tag every unified transaction with its txn_date
unified_dated = unified_df.withColumn("txn_date", F.to_date("txn_ts"))

# ── High-value path ───────────────────────────────────────────────────────────
high_value_txns = (
    unified_dated
    .join(daily_customer_totals, on=["customer_id", "txn_date"], how="inner")
    .withColumn("sar_reason", F.lit("DAILY_THRESHOLD_BREACH"))
    .withColumn("fraud_rule_triggered", F.lit(None).cast("array<string>"))
    .select(
        "txn_id", "customer_id", "txn_channel", "txn_amount_inr",
        "txn_ts", "txn_date", "status", "merchant_id",
        "is_international", "fraud_rule_triggered", "sar_reason"
    )
)

# ── Fraud-flagged path ────────────────────────────────────────────────────────
fraud_txns = (
    fraud_df
    .withColumn("txn_date", F.to_date("txn_ts"))
    .withColumn("sar_reason", F.lit("FRAUD_RULE_TRIGGERED"))
    .select(
        "txn_id", "customer_id", "txn_channel", "txn_amount_inr",
        "txn_ts", "txn_date", "status", "merchant_id",
        "is_international", "fraud_rule_triggered", "sar_reason"
    )
)

# ── Union both paths ──────────────────────────────────────────────────────────
sar_combined = high_value_txns.unionByName(fraud_txns)

# ── Dedup: prefer fraud row (has rule detail) + prefer non-null merchant_id ───
dedup_window = Window.partitionBy("txn_id").orderBy(
    F.when(F.col("sar_reason") == "FRAUD_RULE_TRIGGERED", 0).otherwise(1),
    F.when(F.col("merchant_id").isNotNull(), 0).otherwise(1)
)

sar_deduped = (
    sar_combined
    .withColumn("_rn", F.row_number().over(dedup_window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

# ── Enrich with risk_rating ───────────────────────────────────────────────────
# CARD: customer_id = raw CUST000xxxx → joins to customers_masked directly ✅
# UPI/NEFT: customer_id = hash of payer_vpa/debtor_name → no customer master
#           linkage possible (upstream Task 2.2 limitation) → default UNKNOWN
customers_risk = (
    spark.table(CUSTOMERS_MASKED)
    .select("customer_id", "risk_rating")
    .dropDuplicates(["customer_id"])
)

sar_enriched = (
    sar_deduped
    .join(customers_risk, on="customer_id", how="left")
    .withColumn(
        "risk_rating",
        F.when(F.col("risk_rating").isNull(), F.lit("UNKNOWN"))
         .otherwise(F.col("risk_rating"))
    )
)

# ── Final SAR table — ZERO PII, only masked/tokenized identifiers ─────────────
sar_final = (
    sar_enriched
    .withColumn("report_date",      F.lit(REPORT_DATE).cast(DateType()))
    .withColumn("masked_txn_id",    F.sha2(F.col("txn_id"), 256))
    .withColumn("customer_token",   F.sha2(F.col("customer_id"), 256))
    .withColumn("reporting_branch", F.lit("CENTRAL_COMPLIANCE"))
    .withColumn("_gold_load_ts",    F.lit(JOB_TS).cast(TimestampType()))
    .select(
        "report_date",
        "masked_txn_id",
        "customer_token",
        "txn_amount_inr",
        "txn_channel",
        "fraud_rule_triggered",
        "risk_rating",
        "reporting_branch",
        "sar_reason",
        "is_international",
        "status",
        "merchant_id",   # null for UPI/NEFT_RTGS — expected, no merchant concept
        "txn_ts",
        "_gold_load_ts"
    )
)

sar_count = sar_final.count()

# ── Final null audit ──────────────────────────────────────────────────────────
null_risk_final    = sar_final.filter(F.col("risk_rating") == "UNKNOWN").count()
null_merchant_card = sar_final.filter(
    (F.col("txn_channel") == "CARD") & F.col("merchant_id").isNull()
).count()

print(f"📋 SAR records             : {sar_count:,}")
print(f"   risk_rating UNKNOWN     : {null_risk_final:,}  (UPI/NEFT — no customer master link)")
print(f"   CARD merchant_id nulls  : {null_merchant_card}  {'✅' if null_merchant_card == 0 else '❌ unexpected'}")
print()
sar_final.groupBy("txn_channel", "risk_rating").count().orderBy("txn_channel", "risk_rating").show(truncate=False)
sar_final.show(5, truncate=False)

In [0]:
sar_final.display()

In [0]:
# ── Write SAR table partitioned by report_date ────────────────────────────────
(
    sar_final.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"report_date = '{REPORT_DATE}'")
    .partitionBy("report_date")
    .saveAsTable(SAR_TABLE)
)

print(f"✅ SAR report written — {sar_count:,} rows → {SAR_TABLE}")
print(f"   Partition: report_date = {REPORT_DATE}")
print(f"   SLA target: 06:50 AM | Job completed: {datetime.now().strftime('%H:%M:%S')}")

In [0]:
spark.createDataFrame([{
    "run_id"        : f"task_3_1_{JOB_TS.strftime('%Y%m%d_%H%M%S')}",
    "check_type"    : "AUDIT",
    "source_system" : "silver_layer.unified_transactions,silver_layer.fraud_alerts",
    "check_name"    : "SAR_REPORT_ROW_COUNT",
    "passed"        : True,
    "detail"        : (
        f"report_date={REPORT_DATE} | "
        f"high_value_customer_days={high_value_customers_count} | "
        f"sar_rows_written={sar_count} | "
        f"zero_pii=true"
    ),
    "check_ts"      : JOB_TS
}]).write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.abc_audit_log")

print("✅ ABC audit log updated for Task 3.1")